# Human-in-the-Loop: предварительные проверки, ранжирование по уровню риска и ведение аудита

В README к этому уроку вводится концепция Human-in-the-Loop с коротким фрагментом, который предлагает пользователю выбрать `APPROVE` или `REJECT` после того, как агент уже сгенерировал ответ. Этот подход является хорошей отправной точкой, но производственные реализации HITL обычно требуют три дополнительных элемента:

1. **Проверка перед действием**, которая выполняется **до** того, как агент приступит к выполнению рискованного шага, чтобы контролировать затраты, необратимость и задержки.
2. **Ранжирование по уровню риска**, при котором низкорисковые действия выполняются автоматически, среднерисковые — одобряются пакетно, а только высокорисковые требуют вмешательства человека.
3. **Ведение аудита и цикл доработки**, при котором каждое решение по проверке сохраняется в формате JSONL, а отклонение инициирует повторный запрос к агенту с структурированной причиной, а не просто выводит `Revising...`.

В этом ноутбуке реализованы все эти элементы поверх тех же примитивов, что и в `06-system-message-framework.ipynb`. Он запускается сквозным режимом в `DEMO_MODE = True` (без необходимости интерактивного ввода) или с реальными `input()` запросами при `DEMO_MODE = False`. Примечание: в DEMO_MODE повторная попытка третьей цели запрограммирована скриптом, чтобы механика цикла была демонстративной. Реализация доработки с пере-классификацией требует `DEMO_MODE = False` и участия оператора.

**Вне сферы данного урока (рассматривается в других уроках):** аутентификация и управление доступом (угроза 2 из README урока 06), промежуточное ПО для вызова инструментов (углубление в MAF в уроке 14), паттерны дебатов между агентами.

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

token = os.environ.get("GITHUB_TOKEN", "")
if not token:
    raise RuntimeError(
        "GITHUB_TOKEN environment variable is not set. This notebook needs "
        "a GitHub PAT with 'Models: Read-only' permission. Set GITHUB_TOKEN "
        "in your environment or a local .env file before running."
    )

endpoint = "https://models.github.ai/inference"
model_name = "gpt-4o"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


## Pattern 1: Контроль до действия

Фрагмент HITL из README сначала вызывает агента, а затем просит пользователя утвердить вывод. Это **поток после действия**. Агент уже выполнил операцию, поэтому стоимость вызова LLM уже оплачена, и любой побочный эффект (отправленное письмо, записанная строка в базу данных, опубликованный комментарий) уже произошел.

**Поток до действия** вставляет контроль перед выполнением рискованного шага агентом. Агент предлагает действие, контроль решает, выполнять ли его, и только после одобрения побочный эффект происходит.

| Аспект | Одобрение после действия (фрагмент README) | Контроль до действия (в этом блокноте) |
|---|---|---|
| Когда выполняется одобрение? | После того, как агент сгенерировал вывод | Перед выполнением любого побочного эффекта |
| Стоимость LLM при отказе | Уже оплачена | Оплачивается только за предложение, а не за действие |
| Необратимые побочные эффекты | Возможны (действие уже произошло) | Предотвращены |
| Ясность аудита | Одобрение — это вывод в консоль | Одобрение — это JSON-запись с временной меткой, действием, причиной |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Pattern 2: Ранжирование по уровню риска

Не каждое действие требует одобрения человека. Запрос только для чтения к публичному API отличается по важности от отправки письма клиенту. Относиться к ним одинаково означает тратить внимание оператора впустую и замедлять агента.

Простая модель из 3 уровней:

| Уровень | Примеры | Процесс утверждения |
|---|---|---|
| `низкий` (только для чтения) | Поиск в базе знаний, выбор вариантов рейсов, загрузка публичной веб-страницы | Автоматическое выполнение, логируется для аудита |
| `средний` (незначительное изменение) | Кэширование результата, увеличение счетчика, назначение напоминания | Автоматическое выполнение, но с ежедневным обзором |
| `высокий` (внешнее воздействие или необратимое действие) | Отправка письма, списание денег с карты, публикация в публичном канале | Блокировка до одобрения человеком |

Это один из вариантов ранжирования. В продуктивных системах часто используют более детализированные уровни (например, уровни разрешений AWS IAM, уровни доступа на основе ролей). Версия из 3 уровней ниже — это минимально полезный вариант для агента, который совмещает действия только для чтения и с побочными эффектами.

Классификатор ниже использует эвристику по ключевым словам, чтобы демонстрация оставалась детерминированной и дешевой. В продуктивной системе вы бы заменили его на обученный классификатор или движок политик.

In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Шаблон 3: Журнал аудита и цикл пересмотра

`print("Response approved.")` — это не журнал аудита. Для доверия каждое решение по воротам должно записываться в виде структурированного события, которое можно впоследствии запросить, воспроизвести или приложить к обзору инцидента.

Два элемента:

1. **Добавляющийся только JSONL.** Одна строка на решение с отметкой времени, действием, уровнем, решением, причиной. Легко обрабатывать через grep, легко отправлять в настоящий журнал позже.
2. **Цикл пересмотра при отклонении.** Когда ворота возвращают `deny`, агент повторно запрашивает себя с причиной отклонения в контексте, чтобы следующее предложение могло избежать проблемы.

In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.complete(
        model=model_name,
        messages=[SystemMessage(content=system), UserMessage(content=user_text)],
    )
    return response.choices[0].message.content.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}


In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## Дополнительные ресурсы

Несколько других публичных проектов реализуют вариации этих паттернов HITL. Сравните подходы, чтобы найти то, что подходит для вашего стека:

- **LangChain** обертки инструментов с участием человека в цикле ([документация](https://python.langchain.com/docs/integrations/tools/human_tools)): обертки инструментов, которые приостанавливают выполнение для ввода человеком.
- **AutoGen** `UserProxyAgent` ([документация v0.2](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); в AutoGen v0.4+ была реорганизована): использует роль агента специально для представления человека в многоагентных беседах.
- **Semantic Kernel** фильтры функций ([документация](https://learn.microsoft.com/en-us/semantic-kernel/concepts/enterprise-readiness/filters)): промежуточное ПО, которое запускается вокруг каждого вызова функции, подходит для логики ограничения.

Каждый проект по-разному обрабатывает три подпаттерна: LangChain оборачивает их как инструменты, AutoGen использует роль агента, Semantic Kernel применяет фильтры промежуточного ПО. Прочитайте одну или две реализации полностью, прежде чем выбирать дизайн для собственного агента.

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от ответственности**:
Этот документ был переведен с использованием сервиса машинного перевода [Co-op Translator](https://github.com/Azure/co-op-translator). Несмотря на наши усилия по обеспечению точности, имейте в виду, что автоматический перевод может содержать ошибки или неточности. Оригинальный документ на его исходном языке следует считать авторитетным источником. Для получения критически важной информации рекомендуется обратиться к профессиональному человеческому переводу. Мы не несем ответственности за любые недоразумения или неправильные толкования, возникшие в результате использования этого перевода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
